# 🏷️ NB-04: BERT Multi-Class Topic Classification
**Goal:** Classify user queries into topic categories using BERT.

**Modality:** Text Classification | **Model:** bert-base-uncased

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Auto-label topics from keywords
TOPICS = {
    "coding": ["code","python","implement","function","algorithm","debug","script"],
    "science": ["physics","chemistry","biology","quantum","molecule","cell"],
    "math": ["calculate","equation","proof","theorem","integral","matrix"],
    "philosophy": ["consciousness","ethics","meaning","truth","metaphysics"],
    "ai_ml": ["neural","machine learning","model","training","AI","deep learning"],
    "other": []
}

def label(text):
    text = text.lower()
    for topic, kws in TOPICS.items():
        if any(k in text for k in kws):
            return topic
    return "other"

labels = [label(d["user"]) for d in data]
texts  = [d["user"] for d in data]
le = LabelEncoder()
y = le.fit_transform(labels)
print("Classes:", le.classes_)

from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch, numpy as np
from sklearn.metrics import accuracy_score

tok = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(le.classes_))

def encode(batch):
    enc = tok(batch["text"], truncation=True, padding="max_length", max_length=128)
    return enc

ds = Dataset.from_dict({"text": texts, "label": y.tolist()}).map(encode, batched=True).train_test_split(0.15)

def metrics(p): return {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))}

args = TrainingArguments("./bert-topic", num_train_epochs=4,
    per_device_train_batch_size=16, fp16=torch.cuda.is_available(),
    evaluation_strategy="epoch", load_best_model_at_end=True, report_to="none")
Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
        compute_metrics=metrics).train()
print("✅ BERT classifier ready!")
